# Fly x0 grid on a Colab GPU

Runtime: **T4 GPU** (Runtime → Change runtime type). Run the cells in order.
**Disconnect the runtime when done**: an idle runtime keeps using compute units.


In [1]:
# 1. Code and packages (Colab already has CuPy; flybrain is installed without its GPU extra)
%cd /content
!git clone -q -b colab-experiments --depth 1 https://github.com/LxCenady/Fly-Is-All-You-Need.git
!pip install -q https://github.com/LxCenady/Fly-Is-All-You-Need/releases/download/flybrain-0.1.0.post1/flybrain-0.1.0.post1-py3-none-any.whl
!pip install -q -e Fly-Is-All-You-Need/uctf
import os
os.environ['FLY_REPO'] = '/content/Fly-Is-All-You-Need'
os.environ['PY'] = 'python'
%cd /content/Fly-Is-All-You-Need/experiments/connectome_compare
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for uctf (pyproject.toml) ... done
/content/Fly-Is-All-You-Need/experiments/connectome_compare
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
# 2. Optional: keep results on Google Drive (asks you to allow access)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/fly_colab'
os.makedirs(OUT, exist_ok=True)

In [2]:
# 3. Smoke test: one fly simulation (downloads ~260 MB of connectome data the first time).
#    Expect readout_active_mean near 0.079 (the local run gave 0.0789).
!time python simulate.py specs/fly.json --gain 4.5 --code 3 --shuffle 0 --device cuda --out features_q/smoke_c3.npz | tail -1

flybrain: no brain files in /root/fly-data; downloading the prebuilt brain (~260 MB, once)
downloading brain.npz
  53 / 53 MB
downloading weights.npz
  205 / 205 MB
Traceback (most recent call last):
  File "/content/Fly-Is-All-You-Need/experiments/connectome_compare/simulate.py", line 77, in <module>
    main()
    ~~~~^^
  File "/content/Fly-Is-All-You-Need/experiments/connectome_compare/simulate.py", line 72, in main
    np.savez_compressed(a.out, X=X, info=json.dumps(info))
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 753, in savez_compressed
    _savez(file, args, kwds, True)
    ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 779, in _savez
    zipf = zipfile_factory(file, mode="w", compression=compression)
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 103, in zipfile_factory
    return zipfi

In [ ]:
# 4. The grid (x0_fly.list: 11 conditions x 3 codes, roughly 20 min per condition on a T4).
#    Reruns skip finished files, so it can be resumed after a disconnect.
!bash grid.sh specs/fly.json x0fly cuda x0_fly.list

In [3]:
!git -C /content/Fly-Is-All-You-Need pull -q; git -C /content/Fly-Is-All-You-Need log --oneline -1

afbf98d (HEAD -> colab-experiments, origin/colab-experiments) experiments: simulate.py creates the output folder; notebook setup cell can rerun


In [ ]:
# 5. Analysis (spike counts only, M = 300 and 1000) and copy of the small outputs to Drive
!python analyze.py features_q/x0fly_*.npz --sizes 300,1000 --draws 3 --no-bpc --features spikes > results_x0_fly.json
!python x0_table.py results_x0_fly.json --prefix x0fly_
!cp results_x0_fly*.json logs_q/x0fly_* "$OUT" 2>/dev/null; ls "$OUT" | head